In [1]:
from buildingmotif import BuildingMOTIF
from rdflib import Namespace
from buildingmotif.dataclasses import Model, Library
from buildingmotif.namespaces import BRICK, RDF

bm = BuildingMOTIF("sqlite://") # in-memory instance
# create the namespace
BLDG = Namespace('urn:bldg/') # a unique urn that will be use as the base namespace for other created model
model = Model.create(BLDG, description="This is a test model for a simple building") 
model.graph.add((BLDG["zone-temp"], RDF.type, BRICK.Zone_Air_Temperature_Sensor)) # adding a class manually
print(model.graph.serialize()) # you will see two objects created

# the better way to build brick model is to use templates
# get template
brick_ttl_path = '/home/chenkianwee/kianwee_work/code_workspace/37.buildingmotif/buildingmotif/libraries/brick/Brick.ttl'
brick = Library.load(ontology_graph=brick_ttl_path)
ahu_template = brick.get_template_by_name(BRICK.AHU)
# print template parameters
print("The template has the following parameters:")
for param in ahu_template.parameters:
    print(f"  {param}")

@prefix brick: <https://brickschema.org/schema/Brick#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

<urn:bldg/> a owl:Ontology ;
    rdfs:comment "This is a test model for a simple building" .

<urn:bldg/zone-temp> a brick:Zone_Air_Temperature_Sensor .




/home/chenkianwee/venv/buildingmotif/lib/python3.12/site-packages/pyshacl/extras/__init__.py:46: Warning: Extra "js" is not satisfied because requirement pyduktape2 is not installed.
  warn(Warning(f"Extra \"{extra_name}\" is not satisfied because requirement {req} is not installed."))
2026-04-06 14:09:32,207 | root |  WARNING: An ontology could not resolve a dependency on http://qudt.org/2.1/schema/shacl/qudt (Name: http://qudt.org/2.1/schema/shacl/qudt). Check this is loaded into BuildingMOTIF
2026-04-06 14:09:32,209 | root |  WARNING: An ontology could not resolve a dependency on https://brickschema.org/schema/Brick/ref (Name: https://brickschema.org/schema/Brick/ref). Check this is loaded into BuildingMOTIF
2026-04-06 14:09:32,209 | root |  WARNING: An ontology could not resolve a dependency on https://w3id.org/rec (Name: https://w3id.org/rec). Check this is loaded into BuildingMOTIF
2026-04-06 14:09:32,211 | root |  WARNING: An ontology could not resolve a dependency on http://qud

The template has the following parameters:
  name


In [23]:
ahu_name = "Core_ZN-PSC_AC"
ahu_binding_dict = {"name": BLDG[ahu_name]}
ahu_graph = ahu_template.evaluate(ahu_binding_dict)
model.add_graph(ahu_graph)
# ahu_graph is just an instance of rdflib.Graph
print(ahu_graph.serialize())

@prefix brick: <https://brickschema.org/schema/Brick#> .

<urn:bldg/Core_ZN-PSC_AC> a brick:AHU .




In [24]:
# templates
oa_ra_damper_template = brick.get_template_by_name(BRICK.Outside_Damper)
damper_template = brick.get_template_by_name(BRICK.Damper)
fan_template = brick.get_template_by_name(BRICK.Supply_Fan)
clg_coil_template = brick.get_template_by_name(BRICK.Cooling_Coil)

print(oa_ra_damper_template)
print(damper_template)
print(fan_template)
print(clg_coil_template)

Template(_id=468, _name='https://brickschema.org/schema/Brick#Outside_Damper', body=<Graph identifier=6be15e55-432b-45e6-ae74-d4b38b14e551 (<class 'rdflib.graph.Graph'>)>, optional_args=[], _bm=<buildingmotif.building_motif.building_motif.BuildingMOTIF object at 0x7ea510e50fb0>)
Template(_id=274, _name='https://brickschema.org/schema/Brick#Damper', body=<Graph identifier=6d868e9c-925a-43e7-b45a-aed3d185701b (<class 'rdflib.graph.Graph'>)>, optional_args=[], _bm=<buildingmotif.building_motif.building_motif.BuildingMOTIF object at 0x7ea510e50fb0>)
Template(_id=1210, _name='https://brickschema.org/schema/Brick#Supply_Fan', body=<Graph identifier=b84a1f54-b24e-494f-879a-928c6a6768ee (<class 'rdflib.graph.Graph'>)>, optional_args=[], _bm=<buildingmotif.building_motif.building_motif.BuildingMOTIF object at 0x7ea510e50fb0>)
Template(_id=161, _name='https://brickschema.org/schema/Brick#Cooling_Coil', body=<Graph identifier=0fc08c05-112f-45e0-b35d-c38e24c8142f (<class 'rdflib.graph.Graph'>)>, o

In [25]:
# add the fan into the graph
fan_name = f"{ahu_name}-Fan"
fan_binding_dict = {"name": BLDG[fan_name]}
fan_graph = fan_template.evaluate(fan_binding_dict)
model.add_graph(fan_graph)

# add outdoor air/return air damper
oa_ra_damper_name = f"{ahu_name}-OutsideDamper"
oa_ra_damper_binding_dict = {"name": BLDG[oa_ra_damper_name]}
oa_ra_damper_graph = oa_ra_damper_template.evaluate(oa_ra_damper_binding_dict)
model.add_graph(oa_ra_damper_graph)

# add other damper
damper_name = f"{ahu_name}-Damper"
damper_binding_dict = {"name": BLDG[damper_name]}
damper_graph = damper_template.evaluate(damper_binding_dict)
model.add_graph(damper_graph)

# add clg coil
clg_coil_name = f"{ahu_name}-Clg_Coil"
clg_coil_binding_dict = {"name": BLDG[clg_coil_name]}
clg_coil_graph = clg_coil_template.evaluate(clg_coil_binding_dict)
model.add_graph(clg_coil_graph)

In [26]:
# connect zone-temp, fan, dampers, and clg coil to AHU
model.graph.add((BLDG[ahu_name], BRICK.hasPoint, BLDG["zone-temp"]))
model.graph.add((BLDG[ahu_name], BRICK.hasPart, BLDG[oa_ra_damper_name]))
model.graph.add((BLDG[ahu_name], BRICK.hasPart, BLDG[damper_name]))
model.graph.add((BLDG[ahu_name], BRICK.hasPart, BLDG[fan_name]))
model.graph.add((BLDG[ahu_name], BRICK.hasPart, BLDG[clg_coil_name]))

<Graph identifier=15ec17d8-4b18-4581-89ce-9e49b9179c29 (<class 'rdflib.graph.Graph'>)>

In [27]:
# you can add triples directly too
model.graph.add((BLDG[oa_ra_damper_name], BRICK.hasPoint, BLDG[oa_ra_damper_name + "Position"]))
model.graph.add((BLDG[oa_ra_damper_name + "Position"], RDF.type, BRICK.Damper_Position_Command))
# print model to confirm components were added and connected
print(model.graph.serialize())


@prefix brick: <https://brickschema.org/schema/Brick#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

<urn:bldg/> a owl:Ontology ;
    rdfs:comment "This is a test model for a simple building" .

<urn:bldg/Core_ZN-PSC_AC> a brick:AHU ;
    brick:hasPart <urn:bldg/Core_ZN-PSC_AC-Clg_Coil>,
        <urn:bldg/Core_ZN-PSC_AC-Damper>,
        <urn:bldg/Core_ZN-PSC_AC-Fan>,
        <urn:bldg/Core_ZN-PSC_AC-OutsideDamper> ;
    brick:hasPoint <urn:bldg/zone-temp> .

<urn:bldg/Core_ZN-PSC_AC-Clg_Coil> a brick:Cooling_Coil .

<urn:bldg/Core_ZN-PSC_AC-Damper> a brick:Damper .

<urn:bldg/Core_ZN-PSC_AC-Fan> a brick:Supply_Fan .

<urn:bldg/Core_ZN-PSC_AC-OutsideDamper> a brick:Outside_Damper ;
    brick:hasPoint <urn:bldg/Core_ZN-PSC_AC-OutsideDamperPosition> .

<urn:bldg/Core_ZN-PSC_AC-OutsideDamperPosition> a brick:Damper_Position_Command .

<urn:bldg/zone-temp> a brick:Zone_Air_Temperature_Sensor .




In [ ]:
# model.graph.serialize(destination='/home/chenkianwee/kianwee_work/code_workspace/37.buildingmotif/buildingmotif/tutorial1_mode.ttl')

<Graph identifier=15ec17d8-4b18-4581-89ce-9e49b9179c29 (<class 'rdflib.graph.Graph'>)>

In [ ]:
for s, p, o in model.graph:
    print(s)

urn:bldg/ http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2002/07/owl#Ontology
urn:bldg/ http://www.w3.org/2000/01/rdf-schema#comment This is a test model for a simple building
urn:bldg/zone-temp http://www.w3.org/1999/02/22-rdf-syntax-ns#type https://brickschema.org/schema/Brick#Zone_Air_Temperature_Sensor
urn:bldg/Core_ZN-PSC_AC-Fan http://www.w3.org/1999/02/22-rdf-syntax-ns#type https://brickschema.org/schema/Brick#Supply_Fan
urn:bldg/Core_ZN-PSC_AC-OutsideDamper http://www.w3.org/1999/02/22-rdf-syntax-ns#type https://brickschema.org/schema/Brick#Outside_Damper
urn:bldg/Core_ZN-PSC_AC-Damper http://www.w3.org/1999/02/22-rdf-syntax-ns#type https://brickschema.org/schema/Brick#Damper
urn:bldg/Core_ZN-PSC_AC-Clg_Coil http://www.w3.org/1999/02/22-rdf-syntax-ns#type https://brickschema.org/schema/Brick#Cooling_Coil
urn:bldg/Core_ZN-PSC_AC-OutsideDamperPosition http://www.w3.org/1999/02/22-rdf-syntax-ns#type https://brickschema.org/schema/Brick#Damper_Position_Command
ur